<a href="https://colab.research.google.com/github/KerellosRezk231/NewProject/blob/main/Ai%20project%20khaled%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [68]:
import pandas as pd

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

df.to_csv('customer_churn_data.csv', index=False)

print(" الداتا جاهزة وفيها 7,043 صف!")
print("تفضل، دي عينة من البيانات:")
df.head()

 الداتا جاهزة وفيها 7,043 صف!
تفضل، دي عينة من البيانات:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [69]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

df.drop('customerID', axis=1, inplace=True)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)

le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(" تم تنظيف الداتا وتحويلها لأرقام بالكامل.")

 تم تنظيف الداتا وتحويلها لأرقام بالكامل.


In [70]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    min_samples_leaf=2,
    criterion='entropy',
    random_state=42
),
    "SVM": SVC()
}

print("-" * 30)
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"دقة موديل {name}: {acc*100:.2f}%")

------------------------------
دقة موديل Logistic Regression: 78.54%
دقة موديل Random Forest: 79.46%
دقة موديل SVM: 79.10%


In [71]:
import joblib
from sklearn.preprocessing import StandardScaler

joblib.dump(models["Random Forest"], 'churn_model.pkl')

scaler_model = StandardScaler()
X_rescaled = scaler_model.fit(X)
joblib.dump(scaler_model, 'scaler.pkl')

print(" تم حفظ الموديل والميزان بنجاح!")

 تم حفظ الموديل والميزان بنجاح!


In [72]:
%%writefile app_churn.py
import streamlit as st
import joblib
import numpy as np

model = joblib.load('churn_model.pkl')
scaler = joblib.load('scaler.pkl')

st.set_page_config(page_title="AI Churn Analyzer", layout="wide")
st.title(" نظام تحليل أسباب مغادرة العملاء")

col1, col2, col3 = st.columns(3)
with col1:
    gender = st.selectbox("gender", ["Male", "Female"])
    SeniorCitizen = st.selectbox("SeniorCitizen", [0, 1])
    Partner = st.selectbox("Partner", ["Yes", "No"])
    Dependents = st.selectbox("Dependents", ["Yes", "No"])
    tenure = st.number_input("tenure (Months)", 0, 72, 1)
    PhoneService = st.selectbox("PhoneService", ["Yes", "No"])
with col2:
    MultipleLines = st.selectbox("MultipleLines", ["No phone service", "No", "Yes"])
    InternetService = st.selectbox("InternetService", ["Fiber optic", "DSL", "No"])
    OnlineSecurity = st.selectbox("OnlineSecurity", ["No", "Yes", "No internet service"])
    OnlineBackup = st.selectbox("OnlineBackup", ["No", "Yes", "No internet service"])
    DeviceProtection = st.selectbox("DeviceProtection", ["No", "Yes", "No internet service"])
    TechSupport = st.selectbox("TechSupport", ["No", "Yes", "No internet service"])
with col3:
    StreamingTV = st.selectbox("StreamingTV", ["No", "Yes", "No internet service"])
    StreamingMovies = st.selectbox("StreamingMovies", ["No", "Yes", "No internet service"])
    Contract = st.selectbox("Contract", ["Month-to-month", "One year", "Two year"])
    PaperlessBilling = st.selectbox("PaperlessBilling", ["Yes", "No"])
    PaymentMethod = st.selectbox("PaymentMethod", ["Electronic check", "Mailed check", "Bank transfer", "Credit card"])
    MonthlyCharges = st.number_input("MonthlyCharges", 0.0, 150.0, 110.0)
    TotalCharges = st.number_input("TotalCharges", 0.0, 9000.0, 110.0)

if st.button("تحليل الأسباب"):
    def encode(val):
        mapping = {"Yes": 1, "No": 0, "Male": 1, "Female": 0, "Month-to-month": 0, "One year": 1, "Two year": 2,
                   "DSL": 0, "Fiber optic": 1, "Electronic check": 2, "Mailed check": 3, "Bank transfer": 0, "Credit card": 1,
                   "No internet service": 2, "No phone service": 1}
        return mapping.get(val, 0)

    features = [
        encode(gender), SeniorCitizen, encode(Partner), encode(Dependents),
        tenure, encode(PhoneService), encode(MultipleLines), encode(InternetService),
        encode(OnlineSecurity), encode(OnlineBackup), encode(DeviceProtection),
        encode(TechSupport), encode(StreamingTV), encode(StreamingMovies),
        encode(Contract), encode(PaperlessBilling), encode(PaymentMethod),
        MonthlyCharges, TotalCharges
    ]

    scaled_features = scaler.transform([features])
    prediction = model.predict(scaled_features)
    probability = model.predict_proba(scaled_features)[0][1] * 100

    st.divider()

    if prediction[0] == 1:
        st.error(f" النتيجة: العميل سيغادر الشركة بنسبة {probability:.1f}%")

        st.subheader(" تحليل أسباب المغادرة المحتملة:")
        reasons = []
        if Contract == "Month-to-month":
            reasons.append("• نوع العقد (Month-to-month) يسهل على العميل الرحيل في أي وقت.")
        if InternetService == "Fiber optic":
            reasons.append("• خدمة الـ Fiber optic مرتبطة بمعدل شكاوى عالي وتكلفة مرتفعة.")
        if MonthlyCharges > 70:
            reasons.append(f"• الفاتورة الشهرية ({MonthlyCharges}$) تعتبر مرتفعة مقارنة بالمتوسط.")
        if tenure < 12:
            reasons.append(f"• العميل ما زال في مرحلة البداية ({tenure} شهر) ولم يبنِ ولاءً للشركة بعد.")
        if TechSupport == "No":
            reasons.append("• غياب الدعم الفني (Tech Support) يزيد من إحباط العميل عند حدوث مشاكل.")

        for r in reasons:
            st.write(r)
    else:
        st.success(f" النتيجة: العميل مستمر بنسبة {100-probability:.1f}%")
        st.write(" العميل يبدو مستقراً بسبب التزامه بالعقود أو انخفاض التكلفة.")


Overwriting app_churn.py


In [ ]:
!pip install streamlit -q
!pip install pyngrok -q

import urllib
print("الـ Password هو:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

!streamlit run app_churn.py & npx localtunnel --port 8501

الـ Password هو: 34.42.236.17
⠙

⠹⠸⠼⠴⠦your url is: https://true-signs-visit.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.42.236.17:8501

